# 03 Univariate and bivariate analysis


Distributions, outliers and relationships with the target `readmitted_30` (readmitted within 30 days).
Input: `data/processed/diabetic_data.parquet` from `02_cleaning`. Figures are saved to `reports/figures/`.


In [ ]:
%load_ext autoreload
%autoreload 2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

from eda_diabetes.data import ADMISSION_TYPE, load_processed
from eda_diabetes.paths import REPORTS_FIGURES

sns.set_theme(style="whitegrid")
REPORTS_FIGURES.mkdir(parents=True, exist_ok=True)


def save(fig, name: str) -> None:
    fig.savefig(REPORTS_FIGURES / f"{name}.png", dpi=150, bbox_inches="tight")

In [ ]:
df = load_processed()
df["admission_type"] = df["admission_type_id"].map(ADMISSION_TYPE).fillna("Unknown")
TARGET = "readmitted_30"
print(df.shape)
print(f"base rate {TARGET}: {df[TARGET].mean():.3f}")

In [ ]:
NUMERIC = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_diagnoses",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "total_visits",
    "n_meds_used",
    "n_meds_changed",
]
CATEGORICAL = [
    "age",
    "race",
    "gender",
    "admission_type",
    "discharge_group",
    "admission_source_group",
    "payer_code",
    "medical_specialty",
    "diag_1",
    "max_glu_serum",
    "A1Cresult",
    "insulin",
    "change",
    "diabetesMed",
]

## Univariate: numeric


In [ ]:
summary = df[NUMERIC].describe().T
summary["skew"] = df[NUMERIC].skew()
summary["pct_zero"] = (df[NUMERIC] == 0).mean() * 100
summary.round(2)

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 9))
for ax, col in zip(axes.flat, NUMERIC, strict=False):
    sns.histplot(df[col], ax=ax, discrete=df[col].nunique() < 30)
    ax.set_title(col)
    ax.set_xlabel("")
for ax in axes.flat[len(NUMERIC) :]:
    ax.set_visible(False)
fig.tight_layout()
save(fig, "numeric_distributions")

## Univariate: categorical


In [ ]:
fig, axes = plt.subplots(5, 3, figsize=(16, 20))
for ax, col in zip(axes.flat, CATEGORICAL, strict=False):
    order = df[col].value_counts().index[:12]
    sns.countplot(data=df, y=col, order=order, ax=ax)
    ax.set_title(col)
    ax.set_ylabel("")
for ax in axes.flat[len(CATEGORICAL) :]:
    ax.set_visible(False)
fig.tight_layout()
save(fig, "categorical_distributions")

## Outliers

Flag values outside 1.5 x IQR from the quartiles. Count per column.


In [ ]:
def iqr_outliers(s: pd.Series) -> pd.Series:
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return (s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)


outliers = pd.DataFrame(
    {
        "n_iqr_outliers": {c: int(iqr_outliers(df[c]).sum()) for c in NUMERIC},
        "pct": {c: round(iqr_outliers(df[c]).mean() * 100, 1) for c in NUMERIC},
        "max": df[NUMERIC].max(),
        "p99": df[NUMERIC].quantile(0.99),
    }
)
outliers

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 9))
for ax, col in zip(axes.flat, NUMERIC, strict=False):
    sns.boxplot(x=df[col], ax=ax)
    ax.set_title(col)
    ax.set_xlabel("")
for ax in axes.flat[len(NUMERIC) :]:
    ax.set_visible(False)
fig.tight_layout()
save(fig, "numeric_boxplots")

The visit counts have long tails. Check whether extreme values look like errors or real heavy use.


In [ ]:
df[["number_outpatient", "number_emergency", "number_inpatient"]].apply(
    lambda s: s.value_counts().sort_index().tail(5).to_dict()
)

In [ ]:
# Does the target rate hold up in the tail of each count column?
rows = {}
for col in ["number_outpatient", "number_emergency", "number_inpatient", "num_medications"]:
    cutoff = df[col].quantile(0.99)
    tail = df[col] > cutoff
    rows[col] = {"p99": cutoff, "n_tail": int(tail.sum()), "rate_tail": df.loc[tail, TARGET].mean()}
pd.DataFrame(rows).T.round(3)

### Outlier decision: keep

- The IQR rule flags every nonzero value of the visit counts (87-93% zeros), so the flagged share (7-13%) says little.
- The extremes (42 outpatient or emergency visits in a year, 12 inpatient) are rare but possible heavy use, not clear errors.
- The tail carries signal. Above the 99th percentile of `number_inpatient` the 30-day rate is about 29%, against 9% overall. Dropping the tail would remove the highest-risk patients.
- Decision: keep all rows. Use `log1p` for plots of the visit counts, and a rank-based or tree model later.


## Bivariate: categorical vs target


In [ ]:
MIN_N = 100  # smaller levels give intervals too wide to read
fig, axes = plt.subplots(5, 3, figsize=(16, 20))
for ax, col in zip(axes.flat, CATEGORICAL, strict=False):
    stats_ = df.groupby(col, observed=True)[TARGET].agg(["mean", "size"])
    order = stats_.query("size >= @MIN_N")["mean"].sort_values().index[-12:]
    sns.barplot(data=df, y=col, x=TARGET, order=order, errorbar=("ci", 95), ax=ax)
    ax.axvline(df[TARGET].mean(), color="k", ls="--", lw=1)
    ax.set_title(col)
    ax.set_ylabel("")
    ax.set_xlabel("P(readmitted < 30 days)")
for ax in axes.flat[len(CATEGORICAL) :]:
    ax.set_visible(False)
fig.tight_layout()
save(fig, "target_rate_by_category")

Dashed line: overall rate. Levels with fewer than 100 encounters are left out. An interval crossing the line means no clear difference.


## Bivariate: numeric vs target


In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for ax, col in zip(axes.flat, NUMERIC, strict=False):
    sns.boxplot(data=df, x=TARGET, y=col, showfliers=False, ax=ax)
    ax.set_title(col)
for ax in axes.flat[len(NUMERIC) :]:
    ax.set_visible(False)
fig.tight_layout()
save(fig, "numeric_by_target")

In [ ]:
rows = {}
for col in NUMERIC:
    yes, no = df.loc[df[TARGET] == 1, col], df.loc[df[TARGET] == 0, col]
    u = stats.mannwhitneyu(yes, no)
    rows[col] = {
        "median_<30": yes.median(),
        "median_other": no.median(),
        "mean_<30": yes.mean(),
        "mean_other": no.mean(),
        "p_value": u.pvalue,
        # common-language effect size: P(random <30 case > random other case)
        "prob_superiority": u.statistic / (len(yes) * len(no)),
    }
mw = pd.DataFrame(rows).T.sort_values("prob_superiority", ascending=False)
mw.round(4)

With about 70k rows, tiny differences get small p-values. Read `prob_superiority` (0.5 = no difference) as the effect size.


Effects are small. Longer stay (`time_in_hospital`) and more prior inpatient visits go with 30-day readmission,
but `prob_superiority` stays below 0.56 for every column.


## Bivariate: association of each categorical with target

Chi-square test and Cramer's V (0 = none, 1 = perfect).


In [ ]:
def cramers_v(a: pd.Series, b: pd.Series) -> float:
    table = pd.crosstab(a, b)
    chi2 = stats.chi2_contingency(table)[0]
    n = table.to_numpy().sum()
    k = min(table.shape) - 1
    return float(np.sqrt(chi2 / (n * k))) if k > 0 else 0.0


rows = {}
for col in CATEGORICAL:
    table = pd.crosstab(df[col], df[TARGET])
    chi2, p, dof, _ = stats.chi2_contingency(table)
    rows[col] = {
        "n_levels": table.shape[0],
        "chi2": chi2,
        "p_value": p,
        "cramers_v": cramers_v(df[col], df[TARGET]),
    }
assoc = pd.DataFrame(rows).T.sort_values("cramers_v", ascending=False)
assoc.round(4)

All Cramer's V values are below 0.11. `discharge_group` is the strongest (patients sent to a facility are readmitted more),
then `age` (rate rises up to the 80-90 bin). `gender` has no association.
Levels such as `Newborn` and `Trauma Center` have too few rows to read.


## Multivariate


In [ ]:
corr = df[[*NUMERIC, TARGET]].corr(method="spearman")
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0, ax=ax)
ax.set_title("Spearman correlation")
save(fig, "spearman_heatmap")

In [ ]:
# Strong pairs among predictors (excluding the target).
pairs = corr.drop(index=TARGET, columns=TARGET).stack()
pairs = pairs[pairs.index.get_level_values(0) < pairs.index.get_level_values(1)]
pairs.loc[pairs.abs() > 0.4].sort_values(key=abs, ascending=False).to_frame("spearman")

In [ ]:
key = [
    "age",
    "race",
    "admission_type",
    "discharge_group",
    "medical_specialty",
    "diag_1",
    "A1Cresult",
    "insulin",
    "change",
    TARGET,
]
cv = pd.DataFrame([[cramers_v(df[a], df[b]) for b in key] for a in key], index=key, columns=key)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cv, annot=True, fmt=".2f", cmap="mako_r", vmin=0, vmax=1, ax=ax)
ax.set_title("Cramer's V")
save(fig, "cramers_v_heatmap")

### Interaction: HbA1c test x medication change

Question from the original study (Strack et al., 2014): does measuring HbA1c and changing medication go with lower readmission?


In [ ]:
rate = df.pivot_table(
    index="A1Cresult", columns="change", values=TARGET, aggfunc="mean", observed=True
)
count = df.pivot_table(
    index="A1Cresult", columns="change", values=TARGET, aggfunc="size", observed=True
)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.heatmap(rate, annot=True, fmt=".3f", cmap="rocket_r", ax=axes[0])
axes[0].set_title("P(readmitted < 30 days)")
sns.heatmap(count, annot=True, fmt="d", cmap="Blues", ax=axes[1])
axes[1].set_title("n encounters")
fig.tight_layout()
save(fig, "a1c_by_change")

In [ ]:
# Same question for prior inpatient visits, the strongest numeric signal above.
df["prior_inpatient"] = pd.cut(
    df["number_inpatient"], [-1, 0, 1, 2, np.inf], labels=["0", "1", "2", "3+"]
)
rate = df.pivot_table(
    index="prior_inpatient", columns="age", values=TARGET, aggfunc="mean", observed=True
)
fig, ax = plt.subplots(figsize=(10, 3.5))
sns.heatmap(rate, annot=True, fmt=".2f", cmap="rocket_r", ax=ax)
ax.set_title("P(readmitted < 30 days) by prior inpatient visits and age")
save(fig, "prior_inpatient_by_age")

## Takeaways

- Base rate of 30-day readmission is 9.0%. The classes are imbalanced.
- No single feature separates the classes. The largest effects are `discharge_group` (Cramer's V 0.108), `time_in_hospital` and prior inpatient visits.
- `A1Cresult` (p = 0.061) and `max_glu_serum` (p = 0.076) are not significant at 5%. `gender` is not associated at all.
- `total_visits` is a sum of the three visit columns (Spearman 0.49-0.69 with each). Use it or the parts, not both, in a linear model.
- Outliers are kept (see decision above).
- Association is not causation. Frail patients are both discharged to facilities and readmitted.
